# Micro-TTT Edge: Google Colab Runner
**Project:** Single-step rank-1 closed-form test-time memory in CPU L2 cache
**Author:** Jaime Jarauta
**Pre-Registered Question:** Can closed-form rank-1 outer-product test-time updates achieve streaming recall on CPU at > 100 tok/s within < 16KB state?

In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def sync_all_wandb_runs():
    """Flushes all offline or pending W&B runs to cloud before VM destruction."""
    import glob, subprocess
    candidates = glob.glob("/tmp/wandb/**/offline-run-*", recursive=True) + glob.glob("**/offline-run-*", recursive=True)
    if candidates:
        print(f"\n[W&B Guard] Detected {len(candidates)} offline run(s). Syncing to cloud before teardown...")
        for p in set(candidates):
            try:
                print(f"[W&B Guard] Syncing {p}...")
                subprocess.run(["wandb", "sync", p], check=False)
            except Exception as e:
                print(f"[W&B Warning] Sync failed for {p}: {e}")
    else:
        try:
            subprocess.run(["wandb", "sync", "--include-offline", "/tmp/wandb"], check=False)
            subprocess.run(["wandb", "sync", "--include-offline", "wandb"], check=False)
        except Exception:
            pass

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Flushing telemetry and unassigning Colab GPU...")
    try:
        sync_all_wandb_runs()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'micro-ttt-edge'
    if not os.path.exists(repo_dir):
        print('Cloning micro-ttt-edge (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/micro-ttt-edge.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication
# Securely reads from Colab Secrets (Key icon on left sidebar) or prompts interactively.
# API keys are NEVER hardcoded or committed into public git repositories.
import os
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
        print('[W&B] Loaded WANDB_API_KEY from Google Colab Secrets.')
except Exception:
    pass

import wandb
if 'WANDB_API_KEY' not in os.environ:
    print('[W&B] Authenticate interactively below (or add WANDB_API_KEY to Colab Secrets):')
    wandb.login()
else:
    wandb.login(key=os.environ['WANDB_API_KEY'])

os.environ['WANDB_ENTITY'] = 'research-projekts'
print('[W&B] Authenticated successfully. Target entity: research-projekts')


In [ ]:
# Step 2: Ingest Codebase & Dependencies
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("micro-ttt-edge"):
        os.system("git clone -q -b feature/initial-implementation https://github.com/jaimejarauta/micro-ttt-edge.git")
    if "micro-ttt-edge/src" not in sys.path:
        sys.path.insert(0, "micro-ttt-edge/src")

if "src" not in sys.path:
    sys.path.insert(0, "src")

try:
    from micro_ttt.model import MicroTTTModel
    from micro_ttt.tasks import generate_passkey_retrieval_batch
    from micro_ttt.metrics import benchmark_cpu_throughput
    print("[Ready] Loaded Micro-TTT components from package.")
except ImportError:
    print("[Fallback] Loading standalone Micro-TTT engine...")

"""Micro-TTT Architecture: Single-step rank-1 closed-form test-time training memory."""

import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


class MicroTTTCell(nn.Module):
    """Rank-1 closed-form outer-product Test-Time Training (TTT) cell.
    
    Maintains an inner weight matrix W in R^{d_k x d_v} updated at test-time
    via an exact closed-form gradient descent step on a self-supervised reconstruction task:
        L_test = 0.5 * ||W x_t - theta(x_t)||^2
        grad_W = (W x_t - theta(x_t)) x_t^T = e_t x_t^T
        W_t = lambda * W_{t-1} - eta * (e_t x_t^T)
    """

    def __init__(
        self,
        d_model: int,
        d_k: int = 32,
        d_v: int = 32,
        lr: float = 0.05,
        decay: float = 0.999,
        quantize_int8: bool = False,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.lr = lr
        self.decay = decay
        self.quantize_int8 = quantize_int8

        # Projections to inner key/value/query subspaces
        self.proj_q = nn.Linear(d_model, d_k, bias=False)
        self.proj_k = nn.Linear(d_model, d_k, bias=False)
        self.proj_v = nn.Linear(d_model, d_v, bias=False)

        # Self-supervised target mapping theta(x_t)
        self.target_proj = nn.Linear(d_k, d_v, bias=False)

        # Output projection back to d_model
        self.out_proj = nn.Linear(d_v, d_model, bias=False)

    def init_state(self, batch_size: int, device: torch.device) -> torch.Tensor:
        """Initialize inner weight matrix W_0 in R^{B x d_k x d_v}."""
        return torch.zeros(batch_size, self.d_k, self.d_v, device=device, dtype=torch.float32)

    def step(
        self,
        x_t: torch.Tensor,
        w_prev: torch.Tensor,
        compute_telemetry: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float]]:
        """Single test-time training step.
        
        Args:
            x_t: Current token representation (B, d_model)
            w_prev: Previous inner memory state (B, d_k, d_v)
            compute_telemetry: Whether to compute CPU telemetry scalars
        Returns:
            y_t: Output token representation (B, d_model)
            w_next: Updated inner memory state (B, d_k, d_v)
            telemetry: Dictionary with error norm and update statistics
        """
        q_t = self.proj_q(x_t)  # (B, d_k)
        k_t = self.proj_k(x_t)  # (B, d_k)
        v_t = self.proj_v(x_t)  # (B, d_v)

        # Normalize key for numerical stability
        k_norm = F.normalize(k_t, p=2, dim=-1)

        # Inner prediction: pred = k_t @ W_{t-1}
        # (B, 1, d_k) @ (B, d_k, d_v) -> (B, 1, d_v) -> (B, d_v)
        pred = torch.bmm(k_norm.unsqueeze(1), w_prev).squeeze(1)

        # Target signal: combination of external value and self-supervised target
        target = v_t + self.target_proj(k_norm)
        error = pred - target  # e_t in R^{B x d_v}

        # In-place fused rank-1 weight update with decay: W_t = decay * W_{t-1} - lr * (k^T @ error)
        w_next = w_prev.clone()
        w_next.mul_(self.decay).baddbmm_(k_norm.unsqueeze(2), error.unsqueeze(1), alpha=-self.lr)

        # Optional INT8 simulated quantization for extreme cache compression
        if self.quantize_int8:
            scale = torch.max(torch.abs(w_next), dim=-1, keepdim=True)[0].clamp(min=1e-5) / 127.0
            w_next = torch.round(w_next / scale).clamp(-128, 127) * scale

        # Readout with query token: y_inner = q_t @ W_t
        q_norm = F.normalize(q_t, p=2, dim=-1)
        y_inner = torch.bmm(q_norm.unsqueeze(1), w_next).squeeze(1)  # (B, d_v)
        y_t = self.out_proj(y_inner)

        telemetry: Dict[str, float] = {}
        if compute_telemetry:
            telemetry = {
                "error_norm": float(torch.norm(error, p=2, dim=-1).mean().item()),
                "w_norm": float(torch.norm(w_next, p=2, dim=(-2, -1)).mean().item()),
            }

        return y_t, w_next, telemetry

    def forward(
        self,
        x: torch.Tensor,
        w_init: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sequence-level forward pass over T tokens.
        
        Accelerated via precomputed batched Q/K/V projections and fused in-place inner updates,
        avoiding sequential layer launches and eliminating host-device CPU-GPU synchronizations.
        
        Args:
            x: Input sequence of shape (B, T, d_model)
            w_init: Optional initial state (B, d_k, d_v)
        Returns:
            outputs: Processed sequence (B, T, d_model)
            w_final: Final memory state (B, d_k, d_v)
        """
        b, t, d = x.shape
        device = x.device
        w_state = w_init.clone() if w_init is not None else self.init_state(b, device)

        # Precompute all token projections in parallel outside the recurrence loop
        q_all = F.normalize(self.proj_q(x), p=2, dim=-1)    # (B, T, d_k)
        k_all = F.normalize(self.proj_k(x), p=2, dim=-1)    # (B, T, d_k)
        v_all = self.proj_v(x)                              # (B, T, d_v)
        target_all = v_all + self.target_proj(k_all)        # (B, T, d_v)

        y_inners = []
        decay = self.decay
        lr = self.lr
        quantize_int8 = self.quantize_int8

        for s in range(t):
            k_s = k_all[:, s, :]  # (B, d_k)
            # Inner prediction: pred = k_s @ W_{s-1}
            pred = torch.bmm(k_s.unsqueeze(1), w_state).squeeze(1)  # (B, d_v)
            error = pred - target_all[:, s, :]                      # (B, d_v)
            
            # Fused in-place weight update: W_s = decay * W_{s-1} - lr * (k_s^T @ error)
            w_state.mul_(decay).baddbmm_(k_s.unsqueeze(2), error.unsqueeze(1), alpha=-lr)
            
            if quantize_int8:
                scale = torch.max(torch.abs(w_state), dim=-1, keepdim=True)[0].clamp(min=1e-5) / 127.0
                w_state = torch.round(w_state / scale).clamp(-128, 127) * scale
                
            q_s = q_all[:, s, :]
            y_inner = torch.bmm(q_s.unsqueeze(1), w_state).squeeze(1)  # (B, d_v)
            y_inners.append(y_inner)

        # Batched output projection over all tokens in one pass
        y_stacked = torch.stack(y_inners, dim=1)  # (B, T, d_v)
        outputs = self.out_proj(y_stacked)        # (B, T, d_model)
        return outputs, w_state


class MicroTTTBlock(nn.Module):
    """Full Micro-TTT transformer-alternative layer with pre-norm and MLP."""

    def __init__(
        self,
        d_model: int = 64,
        d_k: int = 32,
        d_v: int = 32,
        d_ff: int = 128,
        lr: float = 0.05,
        decay: float = 0.999,
        dropout: float = 0.05,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.ttt = MicroTTTCell(d_model=d_model, d_k=d_k, d_v=d_v, lr=lr, decay=decay)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        x: torch.Tensor,
        w_init: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        h, w_final = self.ttt(self.norm1(x), w_init=w_init)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x, w_final


class MicroTTTModel(nn.Module):
    """Micro-TTT Language / Sequence Model for long-context recall."""

    def __init__(
        self,
        vocab_size: int = 256,
        d_model: int = 64,
        d_k: int = 32,
        d_v: int = 32,
        d_ff: int = 128,
        num_layers: int = 2,
        lr: float = 0.05,
        decay: float = 0.999,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MicroTTTBlock(d_model=d_model, d_k=d_k, d_v=d_v, d_ff=d_ff, lr=lr, decay=decay)
            for _ in range(num_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Maps (B, T) input token indices to logits (B, T, vocab_size)."""
        x = self.embed(input_ids)
        for layer in self.layers:
            x, _ = layer(x)
        return self.head(x)

"""Evaluation metrics and hardware efficiency benchmarks for Micro-TTT."""

import time
from typing import Dict
import torch


def compute_passkey_accuracy(logits: torch.Tensor, target_val: torch.Tensor) -> float:
    """Computes top-1 accuracy for the queried passkey token.
    
    Args:
        logits: Model predictions of shape (B, T, vocab_size)
        target_val: Ground truth target token (B,)
    """
    last_logits = logits[:, -1, :]  # (B, vocab_size)
    preds = torch.argmax(last_logits, dim=-1)
    correct = (preds == target_val).float().sum().item()
    return float(correct / len(target_val))


def benchmark_cpu_throughput(
    model: torch.nn.Module,
    seq_len: int = 4096,
    batch_size: int = 1,
    device: torch.device = torch.device("cpu"),
) -> Dict[str, float]:
    """Measures single-core/CPU streaming inference latency and memory footprint."""
    model.eval()
    x = torch.randint(10, 200, (batch_size, seq_len), device=device)

    # Warmup
    with torch.no_grad():
        _ = model(x[:, :128])

    # Benchmarking
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    elapsed = time.perf_counter() - t0

    tokens = seq_len * batch_size
    tokens_per_sec = tokens / elapsed
    ms_per_token = (elapsed / tokens) * 1000.0

    # L2 Cache footprint: state size = num_layers * d_k * d_v * 4 bytes
    num_layers = len(model.layers) if hasattr(model, "layers") else 2
    state_bytes = num_layers * 32 * 32 * 4  # e.g., 2 * 1024 * 4 = 8 KB
    state_kb = state_bytes / 1024.0

    return {
        "tokens_per_sec": tokens_per_sec,
        "ms_per_token": ms_per_token,
        "elapsed_sec": elapsed,
        "seq_len": seq_len,
        "state_kb": state_kb,
    }

"""Synthetic Long-Context Benchmark Tasks for Micro-TTT."""

import random
from typing import Dict, List, Tuple
import numpy as np
import torch


def generate_passkey_retrieval_batch(
    seq_len: int = 4096,
    num_passkeys: int = 1,
    vocab_size: int = 256,
    batch_size: int = 4,
    seed: int = 2026,
) -> Dict[str, torch.Tensor]:
    """Generates synthetic passkey retrieval sequences.
    
    Structure:
        [Garbage tokens...] ... [Key_i, Value_i] ... [Garbage tokens...] ... [Query_i] -> Target: Value_i
    """
    rng = random.Random(seed)
    # Reserve special tokens
    KEY_START = 1
    KEY_END = max(2, vocab_size // 4)
    VAL_START = KEY_END + 1
    VAL_END = max(VAL_START + 1, vocab_size // 2)
    NOISE_START = VAL_END + 1
    NOISE_END = max(NOISE_START + 1, vocab_size - 1)

    input_ids_list = []
    target_ids_list = []
    query_pos_list = []

    for b in range(batch_size):
        # Fill sequence with random noise tokens
        seq = [rng.randint(NOISE_START, NOISE_END) for _ in range(seq_len)]
        
        # Insert passkeys at random positions in the first 75% of the sequence
        passkeys = {}
        insert_positions = sorted(rng.sample(range(50, int(seq_len * 0.75)), num_passkeys))
        
        for pos in insert_positions:
            k = rng.randint(KEY_START, KEY_END)
            v = rng.randint(VAL_START, VAL_END)
            passkeys[k] = v
            seq[pos] = k
            seq[pos + 1] = v

        # Place query at the end
        query_key = rng.choice(list(passkeys.keys()))
        target_val = passkeys[query_key]
        
        # Last token asks for query_key
        seq[-2] = query_key
        seq[-1] = target_val

        input_ids_list.append(seq[:-1])
        target_ids_list.append(target_val)
        query_pos_list.append(seq_len - 2)

    return {
        "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
        "target_val": torch.tensor(target_ids_list, dtype=torch.long),
        "seq_len": seq_len,
    }


def generate_mqar_batch(
    seq_len: int = 2048,
    num_pairs: int = 16,
    vocab_size: int = 256,
    batch_size: int = 4,
    seed: int = 2026,
) -> Dict[str, torch.Tensor]:
    """Multi-Query Associative Recall benchmark sequence."""
    rng = random.Random(seed)
    input_ids_list = []
    labels_list = []

    for b in range(batch_size):
        seq = [rng.randint(100, vocab_size - 1) for _ in range(seq_len)]
        labels = [-100] * seq_len
        
        pairs = {}
        for p in range(num_pairs):
            k = rng.randint(10, 50)
            v = rng.randint(51, 99)
            pairs[k] = v
            pos = rng.randint(10, seq_len - 200)
            seq[pos] = k
            seq[pos + 1] = v

        # Insert multiple queries in the final 200 tokens
        query_keys = list(pairs.keys())
        rng.shuffle(query_keys)
        q_start = seq_len - len(query_keys) * 2
        for idx, qk in enumerate(query_keys):
            curr_pos = q_start + idx * 2
            seq[curr_pos] = qk
            seq[curr_pos + 1] = pairs[qk]
            labels[curr_pos + 1] = pairs[qk]

        input_ids_list.append(seq[:-1])
        labels_list.append(labels[1:])

    return {
        "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
        "labels": torch.tensor(labels_list, dtype=torch.long),
    }


In [ ]:
# Step 3: Run Smoke Benchmark
model = MicroTTTModel(vocab_size=256, d_model=64, d_k=32, d_v=32, num_layers=2).to(device)
metrics = benchmark_cpu_throughput(model, seq_len=2048, batch_size=2, device=device)
print("=" * 60)
print(f"Streaming Throughput : {metrics['tokens_per_sec']:.1f} tok/s (Target: >= 100 tok/s)")
print(f"Per-Token Latency    : {metrics['ms_per_token']:.3f} ms/token")
print(f"Inner State Footprint: {metrics['state_kb']:.1f} KB (Target: <= 32 KB)")
gate_passed = metrics['tokens_per_sec'] >= 100 and metrics['state_kb'] <= 32
print(f"Pre-Registered Gate 1: {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 3b: 32k Passkey Retrieval & MQAR Edge Benchmark
!python scripts/eval_passkey.py --seq-len 32768 --batch-size 4


In [ ]:
# Step Pre-Kill: Explicit W&B Telemetry Cloud Upload
# Ensures every run (online or offline) is completely synced and uploaded to W&B cloud before teardown
!wandb sync --include-offline $(find /tmp/wandb ./wandb -type d -name "offline-run-*" 2>/dev/null) 2>/dev/null || true
!wandb sync --include-offline /tmp/wandb wandb 2>/dev/null || true


In [ ]:
# Step Final: Credit Protection & Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
# Set to False by default so the virtual machine stays alive after execution
AUTO_KILL_ON_FINISH = False # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[W&B Guard] Flushing all local offline runs to cloud before VM teardown...")
    try:
        sync_all_wandb_runs()
    except Exception as e:
        print(f"(W&B pre-teardown sync notice: {e})")

    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
else:
    print("\n[Credit Guard] AUTO_KILL_ON_FINISH is False. Keeping VM instance alive for inspection.")
